# HuntCrew: agent-led threat detection and response

A crew of agents (CrewAI + RAG over MITRE ATT&CK) detects attacks in **replayed real network traffic**
(CIC-IDS2017), names the technique, plans a policy-checked response, asks a human before risky actions and
writes an incident report. **Grading is done by code against the dataset's own labels**, never by the agents.

Architecture and decisions: see `docs/` in the repository. Run all cells top to bottom (about 10-15 minutes).
Requires a Colab Secret named `MISTRAL_API_KEY` with notebook access enabled.

In [ ]:
!git clone -q https://github.com/Paritsingla7/HuntCrew.git 2>/dev/null || (cd HuntCrew && git pull -q)
%cd HuntCrew
!pip install -q -r requirements.txt

In [ ]:
# Fetch the two external datasets (about 425 MB) with checksums; they are never stored in the repository.
!python -I scripts/fetch_data.py

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from huntcrew import demo

d = demo.setup()          # loads data, builds the ATT&CK knowledge base, wires agents (key from Colab Secrets)
print("model:", d.cfg["llm"]["model"], "| ATT&CK chunks:", len(d.kb.chunks))

## 1. Replay a window of real traffic through the detector
Windows are revealed one at a time; agents can look back but never forward, and never see labels.

In [ ]:
import pandas as pd

truth = demo.windows_labelled(d, "test", "ssh_patator")        # grader's view, only to pick windows to show
pick = sorted(truth.window_index.unique())[:2]
results, incidents = demo.detect(d, "test", "ssh_patator", windows=pick, mode="rules")
for r in results:
    print(f"window {r.window.label()}: {len(r.findings)} verified finding(s), {len(r.rejected)} rejected")
    for f in r.findings:
        print("  ", f.entity, [h.id for h in f.technique_hints], "|", f.behaviour)

## 2. Name the technique, plan the response, and ask before risky actions

In [ ]:
from huntcrew.response import console_approver

outcomes, env = demo.respond(d, incidents, approver=console_approver)   # asks you to approve medium/high risk
for o in outcomes:
    print(o.incident.id, o.incident.entity, "->", [t.id for t in o.assessment.techniques], o.plan.severity)
    for rec in o.log.records:
        print("   ", rec.status.upper(), rec.action.type, rec.action.target, "|", rec.reason)
print("simulated environment:", env.snapshot())

In [ ]:
from IPython.display import Markdown, display

for o in outcomes[:1]:
    display(Markdown(o.report))

## 3. Results (graded by code against dataset labels)

In [ ]:
import json
import pathlib

p = pathlib.Path("results/final.json")
if p.exists():
    res = json.loads(p.read_text())
    rows = []
    for sl, systems in res["slices"].items():
        for sysname, e in systems.items():
            if isinstance(e, dict) and "detection" in e:
                rows.append({"slice": sl, "system": sysname, **{k: e["detection"][k] for k in ("precision", "recall", "f1")},
                             **{k: e.get("response", {}).get(k) for k in ("technique_accuracy_incident", "grounding_rate")}})
    display(pd.DataFrame(rows))
else:
    print("results/final.json not found - run scripts/run_eval.py first")